# OncoVisionAI - Data Exploration

This notebook explores the multimodal cancer detection dataset:
- Image data analysis
- Clinical data statistics
- Data distribution and balance
- Sample visualizations

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 1. Load Clinical Data

In [ ]:
# Load clinical data
clinical_df = pd.read_csv('../data/clinical_data.csv')

print(f"Dataset shape: {clinical_df.shape}")
print(f"\nFirst few rows:")
clinical_df.head()

In [ ]:
# Basic statistics
print("Dataset Statistics:")
print("="*60)
clinical_df.describe()

## 2. Class Distribution

In [ ]:
# Class distribution
class_counts = clinical_df['label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
axes[0].bar(['Benign', 'Malignant'], class_counts.values, color=['green', 'red'], alpha=0.7)
axes[0].set_ylabel('Count', fontweight='bold')
axes[0].set_title('Class Distribution', fontweight='bold', fontsize=14)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
axes[1].pie(class_counts.values, labels=['Benign', 'Malignant'], autopct='%1.1f%%',
           colors=['green', 'red'], startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Class Proportion', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

print(f"\nBenign cases: {class_counts[0]} ({class_counts[0]/len(clinical_df)*100:.1f}%)")
print(f"Malignant cases: {class_counts[1]} ({class_counts[1]/len(clinical_df)*100:.1f}%)")

## 3. Clinical Features Analysis

In [ ]:
# Feature distributions by class
features = ['age', 'symptom_duration_months', 'pain_score', 'lesion_size_mm']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feature in enumerate(features):
    ax = axes[i]
    
    # Plot distributions for both classes
    benign_data = clinical_df[clinical_df['label'] == 0][feature]
    malignant_data = clinical_df[clinical_df['label'] == 1][feature]
    
    ax.hist(benign_data, bins=30, alpha=0.6, label='Benign', color='green')
    ax.hist(malignant_data, bins=30, alpha=0.6, label='Malignant', color='red')
    
    ax.set_xlabel(feature.replace('_', ' ').title(), fontweight='bold')
    ax.set_ylabel('Frequency', fontweight='bold')
    ax.set_title(f'{feature.replace("_", " ").title()} Distribution', fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for feature comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, feature in enumerate(features):
    ax = axes[i]
    
    data_to_plot = [clinical_df[clinical_df['label'] == 0][feature],
                    clinical_df[clinical_df['label'] == 1][feature]]
    
    bp = ax.boxplot(data_to_plot, labels=['Benign', 'Malignant'],
                    patch_artist=True)
    
    # Color boxes
    bp['boxes'][0].set_facecolor('lightgreen')
    bp['boxes'][1].set_facecolor('lightcoral')
    
    ax.set_ylabel(feature.replace('_', ' ').title(), fontweight='bold')
    ax.set_title(f'{feature.replace("_", " ").title()}', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation matrix
correlation_features = features + ['family_history', 'label']
corr_matrix = clinical_df[correlation_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
           square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Sample Image Visualization

In [ ]:
# Load and display sample images
image_dir = Path('../data/raw/images')

# Get sample images (4 benign, 4 malignant)
benign_samples = clinical_df[clinical_df['label'] == 0].sample(4)
malignant_samples = clinical_df[clinical_df['label'] == 1].sample(4)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Display benign samples
for i, (idx, row) in enumerate(benign_samples.iterrows()):
    img_path = image_dir / row['image_id']
    if img_path.exists():
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(img)
        axes[0, i].set_title(f'Benign\nAge: {row["age"]}, Size: {row["lesion_size_mm"]}mm',
                           fontsize=10, color='green', fontweight='bold')
        axes[0, i].axis('off')

# Display malignant samples
for i, (idx, row) in enumerate(malignant_samples.iterrows()):
    img_path = image_dir / row['image_id']
    if img_path.exists():
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[1, i].imshow(img)
        axes[1, i].set_title(f'Malignant\nAge: {row["age"]}, Size: {row["lesion_size_mm"]}mm',
                           fontsize=10, color='red', fontweight='bold')
        axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 6. Summary Statistics by Class

In [ ]:
# Group statistics by class
print("Benign Cases Statistics:")
print("="*60)
print(clinical_df[clinical_df['label'] == 0][features].describe())

print("\n\nMalignant Cases Statistics:")
print("="*60)
print(clinical_df[clinical_df['label'] == 1][features].describe())

## Conclusion

Key findings from data exploration:
1. **Class Balance**: Dataset shows distribution between benign and malignant cases
2. **Feature Patterns**: Malignant cases tend to have higher values for age, duration, pain, and lesion size
3. **Correlations**: Clinical features show meaningful correlations with the target label
4. **Data Quality**: Images and clinical data are properly aligned

This multimodal dataset is suitable for training the OncoVisionAI model!